# Simplicits Debug on Surgical Tissue Point Cloud (Gravity Only)

This notebook loads a **surface point cloud** from your pickle file and runs a **first-pass Simplicits simulation under gravity** (no tool constraints yet).

**Notes**
- This follows the structure of Kaolin's Simplicits Easy API example.
- If you run in **VS Code** and the Kaolin ipywidget visualizer fails, use the **k3d** visualization cells (they work in most environments).


In [1]:
# --- Imports ---
import os
import pickle
import numpy as np
import torch

import kaolin as kal

# For point cloud visualization (recommended for VS Code + Jupyter)
#   pip install k3d
import k3d

print(kal.__file__)
# To get just the directory:
print(os.path.dirname(os.path.abspath(kal.__file__)))

/home/nan/miniconda3/envs/subspace_mfem/lib/python3.10/site-packages/kaolin/__init__.py
/home/nan/miniconda3/envs/subspace_mfem/lib/python3.10/site-packages/kaolin


## 1) Load tissue point cloud from PKL

In [2]:
pkl_file_path = "/home/nan/Desktop/datasets/xpbd/StereoMIS_tissue_tool_trajectories_XPBD/sim_particles/tissue_pts_dnsampled_once.pkl"

with open(pkl_file_path, "rb") as f:
    data = pickle.load(f)

print(type(data), len(data))
print("keys:", data[0].keys())
print("xyz:", np.asarray(data[0]["xyz"]).shape)

<class 'list'> 200
keys: dict_keys(['frame_id', 'xyz', 'rgb', 'opacity', 'indices'])
xyz: (38426, 3)


## 2) Choose a rest frame and move to GPU

In [3]:
# Choose which frame to treat as rest state
rest_frame_idx = 0

xyz = np.asarray(data[rest_frame_idx]["xyz"], dtype=np.float32)  # (N,3)
N = xyz.shape[0]
print("N =", N)

# RAW points (keep these!)
pts_raw = torch.from_numpy(xyz).cuda()

# ---- explicit center/scale (save these for tool normalization!) ----
mn = pts_raw.min(dim=0).values
mx = pts_raw.max(dim=0).values
center_raw = 0.5 * (mn + mx)
half_extent = 0.5 * (mx - mn)
scale_raw = float(half_extent.max().detach().cpu())  # scalar

# Normalize tissue into roughly [-1,1]
pts = (pts_raw - center_raw) / max(scale_raw, 1e-8)

orig_pts = pts.clone()

print("tissue raw bbox min/max:",
      mn.detach().cpu().numpy(),
      mx.detach().cpu().numpy())
print("center_raw:", center_raw.detach().cpu().numpy(), "scale_raw:", scale_raw)
print("tissue normed bbox min/max:",
      pts.min(dim=0).values.detach().cpu().numpy(),
      pts.max(dim=0).values.detach().cpu().numpy())

N = 38426
tissue raw bbox min/max: [-0.76335067 -0.65025973  0.77168   ] [0.5815084  0.41067317 1.8512661 ]
center_raw: [-0.09092113 -0.11979328  1.3114731 ] scale_raw: 0.6724295616149902
tissue normed bbox min/max: [-1.        -0.7888803 -0.8027504] [1.        0.7888803 0.8027503]


## 3) Quick visualization of rest point cloud (k3d)

In [4]:
plot = k3d.plot()
k3d_pts = k3d.points(orig_pts.detach().cpu().numpy(), point_size=0.01)
plot += k3d_pts
plot.display()

Output()

## 4) Material fields (constant for now)

In [5]:
# These are *per-point* material fields used by the elastic loss / simulation.
# Start simple: constant fields.
# Units are not super important for this debug step; tune later.

yms  = torch.full((N,), 2e5, device=pts.device, dtype=pts.dtype)   # Young's modulus
prs  = torch.full((N,), 0.45, device=pts.device, dtype=pts.dtype)  # Poisson ratio
rhos = torch.full((N,), 1000., device=pts.device, dtype=pts.dtype) # Density

# Approx volume: for surface point clouds this is not "true volume".
# Use a rough proxy based on bounding box volume to get reasonable scaling.
mn = pts.min(dim=0).values
mx = pts.max(dim=0).values
bbox_vol = float(torch.prod(mx - mn).detach().cpu())
approx_volume = max(bbox_vol, 1e-6)

print("approx_volume (bbox proxy):", approx_volume)

approx_volume (bbox proxy): 5.06619119644165


## 5) Train a SimplicitsObject (learn weights)

This step learns the **skinning weight field** \(w(x)\) (self-supervised) using elastic energy under random handle transforms.

Start with low iterations to debug the pipeline; increase later.


In [6]:
# Handles = reduced DOFs. Start small for debugging.
num_handles = 5

# Tip: increase training_num_steps after the first end-to-end run works.
sim_obj = kal.physics.simplicits.SimplicitsObject.create_trained(
    pts,
    yms,
    prs,
    rhos,
    approx_volume,
    num_handles=num_handles,

    training_num_steps=3000,
    training_lr_start=1e-3,
    training_lr_end=1e-3,

    # Coeffs similar to Kaolin example; adjust if training unstable.
    training_le_coeff=1e-1,
    training_lo_coeff=1e6,

    training_log_every=500,
    normalize_for_training=True,
)

print("trained object:", sim_obj)

trained object: <kaolin.physics.simplicits.easy_api.SimplicitsObject object at 0x7fe50807f040>


## 6) Build a scene and enable gravity-only simulation

In [7]:
scene = kal.physics.simplicits.SimplicitsScene()

# Quasi-static-ish settings:
scene.max_newton_steps = 50
scene.timestep = 0.01
scene.direct_solve = True

obj_idx = scene.add_object(sim_obj, num_qp=1000)

# Disable gravity for grasping-in-vacuum debug
scene.set_scene_gravity(acc_gravity=torch.tensor([0.0, 0.0, 0.0], device=pts.device, dtype=pts.dtype))

# No floor for now
# scene.set_scene_floor(floor_height=-0.9, floor_axis=1, floor_penalty=1e6)

scene.reset_scene()

internal_pts = scene.get_object_deformed_pts(obj_idx)   # IMPORTANT: no orig_pts
print("internal_pts shape:", internal_pts.shape)


/home/nan/miniconda3/envs/subspace_mfem/lib/python3.10/site-packages/warp/_src/torch.py:280: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  if t.grad is not None:


internal_pts shape: torch.Size([1000, 3])


/home/nan/miniconda3/envs/subspace_mfem/lib/python3.10/site-packages/kaolin/physics/utils/warp_utilities.py:263: UserWarning: Sparse BSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  torch_weights = ctor(


## 7) Tool-driven grasp simulation (quasi-static, no gravity)

This section:
- loads the tool SE(3) trajectory (`tool_3d_poses.pkl`)
- selects fixed **grasped tissue points** (nearest to tool at the first frame)
- applies a **soft attachment** constraint each step via `scene.set_object_boundary_condition(...)`
- steps the simulator and updates the **k3d** visualization.

We start with **translation-only** targets; enable `use_rotation=True` to include tool rotation.


In [8]:
import pickle
import numpy as np
import torch

tool_pkl_path = "/home/nan/Desktop/datasets/xpbd/StereoMIS_tissue_tool_trajectories_XPBD/sim_particles/tool_3d_poses.pkl"
with open(tool_pkl_path, "rb") as f:
    tool_data = pickle.load(f)

frame_ids = sorted(tool_data.keys())
print("tool frames:", frame_ids[0], "->", frame_ids[-1], "count:", len(frame_ids))

tool_t_raw = np.stack([np.asarray(tool_data[k]["t"], dtype=np.float32) for k in frame_ids], axis=0)  # (T,3)
tool_R_raw = np.stack([np.asarray(tool_data[k]["R"], dtype=np.float32) for k in frame_ids], axis=0)  # (T,3,3)

tool_t_raw_t = torch.from_numpy(tool_t_raw).to(device=pts_raw.device, dtype=pts_raw.dtype)
tool_R = torch.from_numpy(tool_R_raw).to(device=pts_raw.device, dtype=pts_raw.dtype)

# IMPORTANT: use RAW tissue normalization params, not orig_pts
# You must have computed these when loading tissue:
# center_raw, scale_raw from pts_raw (raw tissue points)
tool_t = (tool_t_raw_t - center_raw) / max(scale_raw, 1e-8)

print("tool_t range (normed):",
      tool_t.min(dim=0).values.detach().cpu().numpy(),
      tool_t.max(dim=0).values.detach().cpu().numpy())

# sanity: distance from tool to tissue in normalized space
internal0 = scene.get_object_deformed_pts(obj_idx)
dmin = torch.norm(internal0 - tool_t[0][None, :], dim=1).min()
print("min dist tool->tissue (internal) at t0:", float(dmin.detach().cpu()))

tool frames: 75 -> 199 count: 125
tool_t range (normed): [ 0.19295588 -1.1589513  -0.33955088] [ 0.8074347  -0.8500399   0.08211552]
min dist tool->tissue (internal) at t0: 0.28035691380500793


In [9]:
T0 = 0
tool_p0 = tool_t[T0]  # normalized tool translation

# choose grasped points among INTERNAL points
K_GRASP_PTS = 10
d2 = torch.sum((internal_pts - tool_p0[None, :])**2, dim=1)
grasp_ids = torch.topk(d2, k=K_GRASP_PTS, largest=False).indices  # in [0, M)

grasp_mask_cpu = torch.zeros((internal_pts.shape[0],), dtype=torch.bool).cpu()
grasp_mask_cpu[grasp_ids.detach().cpu()] = True

p0 = internal_pts[grasp_ids].clone()   # (K,3) targets are in internal point coordinates
t0 = tool_t[T0].clone()
R0 = tool_R[T0].clone()

def grasp_fcn(_deformed_pts):
    # Must return mask of length M (internal point count)
    return grasp_mask_cpu

In [10]:
# --- Grasp target models ---
K_GRASP = 1e5  # penalty stiffness; tune with timestep/newton
# K_GRASP = 1e7  # penalty stiffness; tune with timestep/newton

def grasp_target_positions_translation_only(t_idx: int) -> torch.Tensor:
    # x_target(t) = p0 + (tool_t(t) - tool_t(t0))
    dt_tool = tool_t[t_idx] - t0
    return p0 + dt_tool[None, :]

def grasp_target_positions_with_rotation(t_idx: int) -> torch.Tensor:
    # Rotate the grasped patch with the tool around the tool origin at t0:
    # x_target(t) = t(t) + R(t) R(t0)^T (p0 - t(t0))
    Rt = tool_R[t_idx]
    tt = tool_t[t_idx]
    return tt[None, :] + (Rt @ (R0.transpose(0, 1) @ (p0 - t0[None, :]).T)).T

In [11]:
dt_norms = torch.norm(tool_t[1:] - tool_t[:-1], dim=1).detach().cpu()
print("tool step norm min/mean/max:", float(dt_norms.min()), float(dt_norms.mean()), float(dt_norms.max()))
print("first step norm:", float(dt_norms[0]))

tool step norm min/mean/max: 0.001601838506758213 0.019682593643665314 0.13565512001514435
first step norm: 0.032657887786626816


In [17]:
import time
import threading
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import k3d

# --------------------------
# 0) Helpers
# --------------------------
def torch_to_np(x):
    return x.detach().cpu().numpy()

# --------------------------
# 1) Create k3d visualization objects
# --------------------------
plot = k3d.plot()

# Surface (high-res) tissue points
surf_np = torch_to_np(scene.get_object_deformed_pts(obj_idx, orig_pts))
k3d_surf = k3d.points(surf_np, point_size=0.01, color=0x0000ff)
plot += k3d_surf

# Internal points (low-res, used for constraints)
internal0 = scene.get_object_deformed_pts(obj_idx)  # (M,3)
M = internal0.shape[0]
internal0_np = torch_to_np(internal0)

# Placeholder grasp/anchor points (will be updated once we build ids)
k3d_grasp = k3d.points(internal0_np[:1], point_size=0.05, color=0xff0000)  # red
k3d_anchor = k3d.points(internal0_np[:1], point_size=0.04, color=0x00ff00) # green

# Tool: current position (as a single point)
tool_np0 = torch_to_np(tool_t[0])[None, :]
k3d_tool = k3d.points(tool_np0, point_size=0.08, color=0xffff00)  # yellow

# Tool trajectory polyline
tool_traj_np = torch_to_np(tool_t)
k3d_tool_traj = k3d.line(tool_traj_np, color=0xffa500, width=0.01)  # orange-ish

# Add overlays (we'll toggle visibility)
plot += k3d_grasp
plot += k3d_anchor
plot += k3d_tool
plot += k3d_tool_traj

plot.display()



# # --------------------------
# # 2) Build anchor + grasp index sets (internal space)
# # --------------------------
# T = len(frame_ids)
# print("Sim steps =", T)
# use_rotation = False

# # Build internal grasp indices once (from existing grasp_mask_cpu)
# # grasp_mask_cpu must be length M
# grasp_ids_internal_cpu = torch.nonzero(torch.tensor(grasp_mask_cpu.numpy()), as_tuple=False).squeeze(1)  # CPU
# grasp_ids_internal_gpu = grasp_ids_internal_cpu.to(device=internal0.device)

# # Anchor patch: farthest from tool at t0
# K_ANCHOR_PTS = 800   # (reduce from 400 for stability)
# tool_p0 = tool_t[0]  # normalized

# d2_anchor = torch.sum((internal0 - tool_p0[None, :])**2, dim=1)
# anchor_ids = torch.topk(d2_anchor, k=K_ANCHOR_PTS, largest=True).indices

# anchor_mask_cpu = torch.zeros((M,), dtype=torch.bool)
# anchor_mask_cpu[anchor_ids.detach().cpu()] = True
# anchor_target = internal0[anchor_ids].clone()

# def anchor_fcn(_):
#     return anchor_mask_cpu

# K_ANCHOR = 1e6

# print("internal0 M:", M, "orig_pts N:", orig_pts.shape[0])
# print("grasp pts:", int(grasp_mask_cpu.sum().item()), "anchor pts:", int(anchor_mask_cpu.sum().item()))

# # Update grasp/anchor viz points now that ids exist
# k3d_grasp.positions = internal0_np[grasp_ids_internal_cpu.numpy()]
# k3d_anchor.positions = internal0_np[anchor_ids.detach().cpu().numpy()]


# --------------------------
# 2) Build anchor + grasp index sets (internal space)
# --------------------------
T = len(frame_ids)
print("Sim steps =", T)
use_rotation = False

# ---- grasp ids (keep your existing grasp_mask_cpu -> grasp ids) ----
grasp_ids_internal_cpu = torch.nonzero(torch.tensor(grasp_mask_cpu.numpy()), as_tuple=False).squeeze(1)  # CPU
grasp_ids_internal_gpu = grasp_ids_internal_cpu.to(device=internal0.device)

# ---- ANCHOR ALL BOUNDARY (EDGE) POINTS ----
# "Edge" proxy for point cloud: near AABB faces in x/y/z.
# Tune eps_ratio: larger => more points anchored.
eps_ratio = 0.03   # 4% of bbox extent (try 0.02~0.08)
mins = internal0.min(dim=0).values
maxs = internal0.max(dim=0).values
extent = (maxs - mins).clamp_min(1e-8)
eps = eps_ratio * extent  # (3,)

# boolean mask on GPU first
on_min_x = (internal0[:, 0] <= mins[0] + eps[0])
on_max_x = (internal0[:, 0] >= maxs[0] - eps[0])
on_min_y = (internal0[:, 1] <= mins[1] + eps[1])
on_max_y = (internal0[:, 1] >= maxs[1] - eps[1])
on_min_z = (internal0[:, 2] <= mins[2] + eps[2])
on_max_z = (internal0[:, 2] >= maxs[2] - eps[2])

anchor_mask_gpu = on_min_x | on_max_x | on_min_y | on_max_y | on_min_z | on_max_z
anchor_mask_gpu[grasp_ids_internal_gpu] = False

# convert to CPU mask because kaolin boundary selector wants CPU bool mask
anchor_mask_cpu = torch.zeros((M,), dtype=torch.bool)
anchor_mask_cpu[torch.nonzero(anchor_mask_gpu, as_tuple=False).squeeze(1).detach().cpu()] = True

anchor_ids = torch.nonzero(anchor_mask_cpu, as_tuple=False).squeeze(1)  # CPU ids
anchor_target = internal0[anchor_ids.to(internal0.device)].clone()      # fixed targets (GPU)

def anchor_fcn(_):
    # IMPORTANT: must return CPU bool tensor length M
    return anchor_mask_cpu

# Strong anchor so boundary doesn't drift
K_ANCHOR = 1e7

print("internal0 M:", M, "orig_pts N:", orig_pts.shape[0])
print("grasp pts:", int(grasp_mask_cpu.sum().item()), "anchor (boundary) pts:", int(anchor_mask_cpu.sum().item()))
print("eps_ratio:", eps_ratio, "eps:", eps.detach().cpu().numpy())

# Update grasp/anchor viz points now that ids exist
k3d_grasp.positions = internal0_np[grasp_ids_internal_cpu.numpy()]
k3d_anchor.positions = internal0_np[anchor_ids.numpy()]


# --------------------------
# 3) UI controls (toggles + stepping)
# --------------------------
ti_state = {"ti": 0}
running = {"flag": False}
runner = {"thread": None}

substeps = widgets.IntSlider(value=1, min=1, max=30, step=1, description="substeps")
sleep_s = widgets.FloatSlider(value=0.03, min=0.0, max=0.5, step=0.01, description="sleep(s)")

btn_next = widgets.Button(description="Next")
btn_run = widgets.Button(description="Run")
btn_stop = widgets.Button(description="Stop")
btn_reset = widgets.Button(description="Reset")

# Toggle overlays
chk_show_grasp  = widgets.Checkbox(value=True,  description="Show grasp pts")
chk_show_anchor = widgets.Checkbox(value=True,  description="Show anchor pts")
chk_show_tool   = widgets.Checkbox(value=True,  description="Show tool pose")
chk_show_traj   = widgets.Checkbox(value=True,  description="Show tool traj")

lbl = widgets.Label()
out = widgets.Output(layout={"border": "1px solid #ddd", "height": "180px", "overflow_y": "auto"})

def apply_visibility():
    k3d_grasp.visible = chk_show_grasp.value
    k3d_anchor.visible = chk_show_anchor.value
    k3d_tool.visible = chk_show_tool.value
    k3d_tool_traj.visible = chk_show_traj.value

chk_show_grasp.observe(lambda _: apply_visibility(), names="value")
chk_show_anchor.observe(lambda _: apply_visibility(), names="value")
chk_show_tool.observe(lambda _: apply_visibility(), names="value")
chk_show_traj.observe(lambda _: apply_visibility(), names="value")

apply_visibility()

def update_vis(ti):
    # Update surface points
    deformed_vis = scene.get_object_deformed_pts(obj_idx, orig_pts)
    k3d_surf.positions = torch_to_np(deformed_vis)

    # Update internal points overlays (grasp/anchor)
    internal_now = scene.get_object_deformed_pts(obj_idx)
    internal_now_np = torch_to_np(internal_now)

    if chk_show_grasp.value:
        k3d_grasp.positions = internal_now_np[grasp_ids_internal_cpu.numpy()]
    if chk_show_anchor.value:
        k3d_anchor.positions = internal_now_np[anchor_ids.detach().cpu().numpy()]

    # Update tool pose point
    if chk_show_tool.value:
        k3d_tool.positions = torch_to_np(tool_t[ti])[None, :]

def apply_constraints_and_step(ti):
    x_target = (grasp_target_positions_with_rotation(ti)
                if use_rotation else
                grasp_target_positions_translation_only(ti))

    # Apply grasp constraint
    scene.set_object_boundary_condition(
        obj_idx=obj_idx, name="grasp",
        fcn=grasp_fcn, bdry_penalty=float(K_GRASP),
        pinned_x=x_target,
    )

    # Apply anchor constraint
    scene.set_object_boundary_condition(
        obj_idx=obj_idx, name="anchor",
        fcn=anchor_fcn, bdry_penalty=float(K_ANCHOR),
        pinned_x=anchor_target,
    )

    # Debug distances (before solve)
    internal_now = scene.get_object_deformed_pts(obj_idx)
    x_now = internal_now[grasp_ids_internal_gpu]
    x_target0 = grasp_target_positions_translation_only(0)
    x_target1 = grasp_target_positions_translation_only(1)

    d0 = torch.norm(x_now - x_target0, dim=1)
    d1 = torch.norm(x_now - x_target1, dim=1)

    # Solve
    scene.run_sim_step()

    # Post-step error to current target
    internal_after = scene.get_object_deformed_pts(obj_idx)
    err = torch.norm(internal_after[grasp_ids_internal_gpu] - x_target, dim=1).mean()

    return float(err.detach().cpu()), float(d0.mean().detach().cpu()), float(d0.max().detach().cpu()), float(d1.mean().detach().cpu()), float(d1.max().detach().cpu())

def do_next(_=None):
    ti = ti_state["ti"]
    if ti >= T:
        lbl.value = "Done."
        return

    with out:
        print(f"\n=== ti={ti} ===")

    err = None
    for ss in range(substeps.value):
        err, d0m, d0M, d1m, d1M = apply_constraints_and_step(ti)
        with out:
            print(f"  substep {ss+1}/{substeps.value} "
                  f"err={err:.6f}  "
                  f"d(now,t0) mean/max={d0m:.6f}/{d0M:.6f}  "
                  f"d(now,t1) mean/max={d1m:.6f}/{d1M:.6f}")

        # Update viz each substep so you can see what happens
        update_vis(ti)

        if sleep_s.value > 0:
            time.sleep(sleep_s.value)

    lbl.value = f"ti={ti:04d}/{T-1}  mean_grasp_err={err:.6f}"
    ti_state["ti"] += 1

def run_loop():
    running["flag"] = True
    while running["flag"] and ti_state["ti"] < T:
        do_next()
    running["flag"] = False

def do_run(_=None):
    if running["flag"]:
        return
    runner["thread"] = threading.Thread(target=run_loop, daemon=True)
    runner["thread"].start()

def do_stop(_=None):
    running["flag"] = False

def do_reset(_=None):
    running["flag"] = False
    scene.reset_scene()
    ti_state["ti"] = 0
    update_vis(0)
    with out:
        print("\n=== RESET ===")
    lbl.value = "Reset."

btn_next.on_click(do_next)
btn_run.on_click(do_run)
btn_stop.on_click(do_stop)
btn_reset.on_click(do_reset)

display(
    widgets.HBox([btn_next, btn_run, btn_stop, btn_reset]),
    widgets.HBox([substeps, sleep_s]),
    widgets.HBox([chk_show_grasp, chk_show_anchor, chk_show_tool, chk_show_traj]),
    lbl,
    out
)

Output()

Sim steps = 125
internal0 M: 1000 orig_pts N: 38426
grasp pts: 10 anchor (boundary) pts: 81
eps_ratio: 0.03 eps: [0.05771643 0.04626874 0.04337823]


Label(value='')

Output(layout=Layout(border='1px solid #ddd', height='180px', overflow_y='auto'))

In [13]:
# # --- Simulation loop: apply grasp constraint + solve next z ---
# deformed = scene.get_object_deformed_pts(obj_idx, orig_pts)
# k3d_pts.positions = deformed.detach().cpu().numpy()

# T = len(frame_ids)
# print("Sim steps =", T)

# use_rotation = False  # set True to include tool rotation

# # sanity
# print("orig_pts device:", orig_pts.device)
# print("grasp_mask_cpu device:", grasp_mask_cpu.device)
# print("tool_t device:", tool_t.device)
# print("x_target device (example):", grasp_target_positions_translation_only(0).device)
# print("grasp_mask_cpu dtype:", grasp_mask_cpu.dtype, "shape:", grasp_mask_cpu.shape)

# for ti in range(T):
#     x_target = (grasp_target_positions_with_rotation(ti)
#                 if use_rotation else
#                 grasp_target_positions_translation_only(ti))

#     # IMPORTANT: this is the grasp constraint hook
#     scene.set_object_boundary_condition(
#         obj_idx=obj_idx,
#         name="grasp",
#         fcn=grasp_fcn,
#         bdry_penalty=float(K_GRASP),
#         pinned_x=x_target,
#     )

#     scene.run_sim_step()

#     deformed = scene.get_object_deformed_pts(obj_idx, orig_pts)
#     k3d_pts.positions = deformed.detach().cpu().numpy()

#     if ti % 10 == 0:
#         err = torch.norm(deformed[grasp_ids] - x_target, dim=1).mean()
#         print(f"t={ti:04d}/{T-1}  mean_grasp_err={float(err.detach().cpu()):.6f}")